# Demo: full context pipeline on one track

Loads the trained checkpoints, takes a single audio file (or a synthetic graph
when no audio is available), and runs the whole system end to end:

* **Task 1/3** predicted tags
* **Task 3** predicted valence and arousal
* **Task 4** top-3 retrieved captions

Runs on **CPU in under two minutes**. Nothing here writes to the results
directory, so it is safe to re-run.

In [1]:
import time

NOTEBOOK_START = time.time()

import json
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import torch

from src.utils import get_device, load_config, resolve_path, set_seed

cfg = load_config(str(ROOT / "config.yaml"))
cfg["device"] = "cpu"          # the demo is a CPU deliverable
device = get_device("cpu")
set_seed(int(cfg.get("seed", 42)))
torch.set_num_threads(4)
print("device:", device)

device: cpu


## 1. Pick an input

Preference order: a real audio file (features are extracted live), otherwise a
pre-built synthetic graph. Set `AUDIO_PATH` to point at any track you like.

In [2]:
AUDIO_PATH = None    # e.g. ROOT / "data/raw/mtat/audio/f/some-clip.mp3"

from src.audio_features import segment_features
from src.graph_builder import build_segment_graph

if AUDIO_PATH and Path(AUDIO_PATH).exists():
    from src.audio_features import load_audio

    t0 = time.time()
    wave = load_audio(AUDIO_PATH, sr=int(cfg["audio"]["sample_rate"]),
                      duration=float(cfg["audio"]["max_duration_s"]))
    feats = segment_features(wave, int(cfg["audio"]["sample_rate"]), cfg)
    graph = build_segment_graph(feats, cfg, track_id=Path(AUDIO_PATH).stem,
                                dataset="demo", split="test", text="")
    source = str(AUDIO_PATH)
    print(f"extracted {feats.shape} features in {time.time() - t0:.1f}s")
else:
    syn_dir = resolve_path(cfg["synthetic"]["out_dir"]) / "graphs"
    candidates = sorted(syn_dir.glob("*.pt"))
    if not candidates:
        raise SystemExit("no audio and no synthetic graphs -- run `python -m src.synthetic`")
    graph = torch.load(candidates[0], weights_only=False)
    source = candidates[0].name
    print("no audio file given; using the synthetic graph", source)

print(f"graph: {graph.num_nodes} segments, {graph.edge_index.shape[1]} edges, "
      f"x={tuple(graph.x.shape)}")

B:\CSE425_Project\gnn-bert-music-context\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


no audio file given; using the synthetic graph syn_0000.pt
graph: 7 segments, 45 edges, x=(7, 96)


In [3]:
from src.graph_builder import visualise_graph
import matplotlib.pyplot as plt

fig = visualise_graph(graph, title=f"input structure graph: {source}")
plt.show()

C:\Users\User\AppData\Local\Temp\ipykernel_19932\321601690.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Load the tag vocabulary and the trained fusion model

In [4]:
from src.bert_encoder import BertTextEncoder, load_tokenizer
from src.fusion_model import GNNBertFusion
from src.gnn_model import GNNEncoder

syn_root = resolve_path(cfg["synthetic"]["out_dir"])
vocab_path = syn_root / "tags.json"
if not vocab_path.exists():
    vocab_path = resolve_path(cfg["paths"]["splits"]) / "tag_vocab.json"
payload = json.loads(Path(vocab_path).read_text(encoding="utf-8"))
TAGS = list(payload["tags"] if isinstance(payload, dict) else payload)
print(f"{len(TAGS)} tags, e.g. {TAGS[:8]}")

ckpt_dir = resolve_path(cfg["paths"].get("checkpoints", "results/checkpoints"))
seed = int(cfg.get("seed", 42))

tokenizer = load_tokenizer(cfg["bert"]["model_name"])
gnn = GNNEncoder(in_dim=int(cfg["graph"]["node_feat_dim"]),
                 hidden_dim=int(cfg["gnn"]["hidden_dim"]),
                 num_layers=int(cfg["gnn"]["num_layers"]),
                 conv=str(cfg["gnn"]["conv"]), dropout=0.0,
                 readout=str(cfg["gnn"]["readout"]))
bert = BertTextEncoder(cfg["bert"]["model_name"], freeze_mode="frozen_probe")
model = GNNBertFusion(gnn, bert, mode=str(cfg["fusion"]["mode"]),
                      shared_dim=int(cfg["fusion"]["shared_dim"]),
                      n_heads=int(cfg["fusion"]["n_heads"]),
                      n_tags=len(TAGS)).to(device).eval()

from src.evaluate import _load_compatible

ckpt = ckpt_dir / f"task3_seed{seed}_best.pt"
if ckpt.exists():
    _load_compatible(model, torch.load(ckpt, map_location=device, weights_only=False)["model_state"])
    thresholds = torch.load(ckpt, map_location="cpu", weights_only=False).get("thresholds")
else:
    print("no Task 3 checkpoint; the numbers below come from an untrained model")
    thresholds = None

50 tags, e.g. ['guitar', 'drum', 'synth', 'piano', 'vocal', 'strings', 'beat', 'slow']


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


03:17:41 | INFO    | gbmc.bert | BERT freeze_mode=frozen_probe -> 0 trainable encoder params (of 6 layers)


03:17:41 | INFO    | gbmc.eval | restored 144/144 tensors (0 incompatible)


## 3. Predict tags and emotion

In [5]:
from torch_geometric.data import Batch

from src.datasets import collate_texts

batch = Batch.from_data_list([graph]).to(device)
ids, mask = collate_texts(batch, tokenizer, int(cfg["bert"]["max_length"]), device)

with torch.no_grad():
    out = model(batch, ids, mask, return_attn=True)

probs = torch.sigmoid(out["tag_logits"][0].float()).cpu().numpy()
thr = np.asarray(thresholds) if thresholds is not None else np.full(len(TAGS), 0.5)
order = np.argsort(-probs)[:10]

print("PREDICTED TAGS  (threshold tuned on validation, frozen)")
print("-" * 52)
for j in order:
    hit = "*" if probs[j] >= thr[j] else " "
    print(f" {hit} {TAGS[j]:<24} p={probs[j]:.3f}  (thr {thr[j]:.2f})")

valence = float(out["valence"][0]); arousal = float(out["arousal"][0])
quadrant = ("high" if valence >= 5 else "low") + " valence / " + \
           ("high" if arousal >= 5 else "low") + " arousal"
print(f"\nVALENCE {valence:.2f} / 9    AROUSAL {arousal:.2f} / 9    ->  {quadrant}")

PREDICTED TAGS  (threshold tuned on validation, frozen)
----------------------------------------------------
 * fast                     p=0.545  (thr 0.53)
 * slow                     p=0.545  (thr 0.51)
 * sitar                    p=0.542  (thr 0.53)
 * orchestra                p=0.542  (thr 0.53)
 * vocal                    p=0.538  (thr 0.52)
 * rhythmic                 p=0.523  (thr 0.51)
 * sparse                   p=0.516  (thr 0.50)
 * violin                   p=0.511  (thr 0.49)
 * brass                    p=0.505  (thr 0.47)
 * distorted                p=0.503  (thr 0.45)

VALENCE 2.67 / 9    AROUSAL 2.41 / 9    ->  low valence / low arousal


In [6]:
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.axhline(5, color="#999", lw=1); ax.axvline(5, color="#999", lw=1)
ax.scatter([valence], [arousal], s=180, color="#c1666b", zorder=3, edgecolors="white")
ax.set_xlim(1, 9); ax.set_ylim(1, 9)
ax.set_xlabel("valence"); ax.set_ylabel("arousal")
for (x, y, label) in [(2.5, 7.5, "tense"), (7.5, 7.5, "excited"),
                      (2.5, 2.5, "sad"), (7.5, 2.5, "calm")]:
    ax.text(x, y, label, ha="center", color="#777", fontsize=9)
ax.set_title("Predicted position in valence-arousal space")
plt.tight_layout(); plt.show()

C:\Users\User\AppData\Local\Temp\ipykernel_19932\2857436037.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 4. Retrieve the top-3 captions (Task 4)

In [7]:
import pandas as pd
import torch.nn.functional as F

from src.contrastive import DualEncoder

dual = DualEncoder(
    GNNEncoder(in_dim=int(cfg["graph"]["node_feat_dim"]),
               hidden_dim=int(cfg["gnn"]["hidden_dim"]),
               num_layers=int(cfg["gnn"]["num_layers"]),
               conv=str(cfg["gnn"]["conv"]), dropout=0.0,
               readout=str(cfg["gnn"]["readout"])),
    BertTextEncoder(cfg["bert"]["model_name"], freeze_mode="frozen_probe"),
    shared_dim=int(cfg["fusion"]["shared_dim"]),
    temperature_init=float(cfg["contrastive"]["temperature_init"]),
).to(device).eval()

ckpt4 = ckpt_dir / f"task4_seed{seed}_best.pt"
if ckpt4.exists():
    _load_compatible(dual, torch.load(ckpt4, map_location=device, weights_only=False)["model_state"])
else:
    print("no Task 4 checkpoint; retrieval below is from an untrained encoder")

# the caption gallery
manifest_path = syn_root / "manifest.csv"
if not manifest_path.exists():
    manifest_path = resolve_path(cfg["paths"]["splits"]) / "musiccaps_manifest.csv"
gallery = pd.read_csv(manifest_path)
gallery = gallery[gallery["split"] == "test"]
gallery = gallery[gallery["text"].astype(str).str.len() > 20].head(200).reset_index(drop=True)
print(f"gallery: {len(gallery)} captions from {manifest_path.name}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


03:17:42 | INFO    | gbmc.bert | BERT freeze_mode=frozen_probe -> 0 trainable encoder params (of 6 layers)


03:17:42 | INFO    | gbmc.eval | restored 125/125 tensors (0 incompatible)


gallery: 29 captions from manifest.csv


In [8]:
t0 = time.time()
with torch.no_grad():
    g_emb = dual.encode_graph(batch)
    texts = gallery["text"].astype(str).tolist()
    chunks = []
    for start in range(0, len(texts), 32):
        enc = tokenizer(texts[start:start + 32], padding=True, truncation=True,
                        max_length=int(cfg["bert"]["max_length"]), return_tensors="pt")
        chunks.append(dual.encode_text(enc["input_ids"].to(device),
                                       enc["attention_mask"].to(device)).cpu())
    t_emb = torch.cat(chunks)

scores = (F.normalize(g_emb.cpu(), dim=-1) @ F.normalize(t_emb, dim=-1).T)[0].numpy()
top3 = np.argsort(-scores)[:3]

print(f"TOP-3 RETRIEVED CAPTIONS   (gallery size {len(gallery)}, "
      f"{time.time() - t0:.1f}s)")
print("=" * 72)
for rank, j in enumerate(top3, start=1):
    print(f"{rank}. score {scores[j]:.3f}  track {gallery.loc[j, 'track_id']}")
    print(f"   {gallery.loc[j, 'text'][:260]}\n")

TOP-3 RETRIEVED CAPTIONS   (gallery size 29, 0.3s)
1. score -0.071  track syn_0134
   A euphoric up-tempo ambient recording featuring guitar, synth, fast, violin. The mix sounds bright and busy.

2. score -0.074  track syn_0182
   A euphoric mid-tempo ambient recording featuring guitar, drum, violin, organ. The mix sounds bright and busy.

3. score -0.074  track syn_0023
   A frantic up-tempo folk recording featuring guitar, drum, synth, slow. The mix sounds muted and busy.



## 5. What the graph attended to

The cross-attention weights over the caption tokens: which words this musical
structure listened to.

In [9]:
attn = out.get("attn")
if attn is not None:
    tokens = tokenizer.convert_ids_to_tokens(ids[0].cpu())
    keep = int(mask[0].sum())
    weights = attn[0].detach().cpu().float().numpy()[:keep]
    tokens = [t.replace("##", "") for t in tokens[:keep]]

    fig, ax = plt.subplots(figsize=(max(6, 0.3 * len(tokens)), 2.4))
    im = ax.imshow(weights.reshape(1, -1), aspect="auto", cmap="viridis")
    ax.set_yticks([])
    ax.set_xticks(range(len(tokens)), tokens, rotation=75, fontsize=7, ha="right")
    ax.set_title("graph query -> caption token cross-attention")
    ax.grid(False)
    fig.colorbar(im, ax=ax, shrink=0.85)
    plt.tight_layout(); plt.show()
else:
    print("this fusion mode exposes no cross-attention map "
          f"(fusion.mode = {cfg['fusion']['mode']})")

C:\Users\User\AppData\Local\Temp\ipykernel_19932\2522092723.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [10]:
elapsed = time.time() - NOTEBOOK_START
print(f"total notebook runtime: {elapsed:.1f}s on {device}")
assert elapsed < 120, "the demo must complete in under 2 minutes on CPU"
print("within the 2-minute CPU budget")

total notebook runtime: 13.5s on cpu
within the 2-minute CPU budget
